## Gas Bottle Baseline Detection

This notebook runs a generic pretrained YOLO model on our local LGP cylinder images.

- Load a pretrained YOLO model in Python
- Run inference on the local image set
- Record what the model detects (COCO classes)
- Show why a pretrained model cannot reliably detect gas cylinders

Outputs: baseline CSV, summary CSV, annotated example images.

- `data/all_images/` — local LGP cylinder images

In [35]:
import os
from pathlib import Path
from collections import Counter
import pandas as pd
import torch
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.14.1+cpu
CUDA available: False


In [36]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

IMAGE_DIR = ROOT / "data" / "all_images"
OUTPUT_DIR = ROOT / "results" / "gas_bottle_baseline"
ANNOTATED_DIR = OUTPUT_DIR / "annotated"
CSV_PATH = OUTPUT_DIR / "baseline_detections.csv"
SUMMARY_PATH = OUTPUT_DIR / "baseline_summary.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_paths = sorted(
    p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in EXTS
)

print(f"Image directory exists: {IMAGE_DIR.exists()}")
print(f"Image directory: {IMAGE_DIR}")
print(f"Found {len(image_paths)} images")

if len(image_paths) == 0:
    raise FileNotFoundError(
        f"No images found under {IMAGE_DIR}. "
        f"Check the folder and file extensions."
    )

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Image directory exists: True
Image directory: d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data\all_images
Found 321 images
Using device: cpu


### Load pretrained YOLO model



In [38]:
model = YOLO("yolo11n.pt")

coco_classes = model.names
has_cylinder_class = any("cylinder" in str(v).lower() for v in coco_classes.values())

print(f"Model classes: {len(coco_classes)}")
print(f"Has explicit 'cylinder' class: {has_cylinder_class}")
print(f"Sample classes: {list(coco_classes.values())[:15]}...")

Model classes: 80
Has explicit 'cylinder' class: False
Sample classes: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird']...


### Run inference on all images

For each image we run prediction at confidence 0.25 and record the number of detections, which classes were detected, and the highest confidence.

In [39]:
records = []

for img_path in image_paths:
    results = model.predict(
        source=str(img_path),
        conf=0.25,
        device=device,
        verbose=False
    )

    result = results[0]
    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        records.append({
            "image": img_path.name,
            "n_detections": 0,
            "classes": "",
            "max_conf": 0.0
        })
        continue

    class_ids = boxes.cls.cpu().numpy().astype(int)
    confs = boxes.conf.cpu().numpy()
    class_names = [coco_classes[i] for i in class_ids]
    unique_classes = sorted(set(class_names))

    records.append({
        "image": img_path.name,
        "n_detections": len(boxes),
        "classes": ";".join(unique_classes),
        "max_conf": float(confs.max())
    })

df = pd.DataFrame(records)
df.to_csv(CSV_PATH, index=False)

print(f"Processed {len(df)} images")
print(f"Saved to {CSV_PATH}")

Processed 321 images
Saved to d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\results\gas_bottle_baseline\baseline_detections.csv


### Baseline statistics

total images, images with at least one detection, total detections, and the class distribution.

In [42]:
total_images = len(df)
images_with_detections = int((df["n_detections"] > 0).sum())
total_detections = int(df["n_detections"].sum())

all_classes = []
for c in df["classes"]:
    if c:
        all_classes.extend(c.split(";"))

class_counts = Counter(all_classes)

print("=" * 50)
print("BASELINE DETECTION STATISTICS")
print("=" * 50)
print(f"Total images:                {total_images}")
print(f"Images with >=1 detection:   {images_with_detections} ({images_with_detections/total_images*100:.1f}%)")
print(f"Images with no detection:    {total_images - images_with_detections}")
print(f"Total detections:            {total_detections}")
print()
print("Detected classes (all occurrences):")
for cls, count in class_counts.most_common():
    print(f"  {cls}: {count}")

BASELINE DETECTION STATISTICS
Total images:                321
Images with >=1 detection:   248 (77.3%)
Images with no detection:    73
Total detections:            454

Detected classes (all occurrences):
  vase: 128
  person: 97
  chair: 32
  bottle: 31
  fire hydrant: 18
  cup: 13
  cake: 12
  wine glass: 9
  toilet: 5
  bowl: 4
  spoon: 4
  car: 4
  keyboard: 3
  bird: 2
  suitcase: 1
  parking meter: 1
  bench: 1
  knife: 1
  orange: 1
  donut: 1


### Save annotated examples

We save a few annotated images: some with detections, some without, to show what the pretrained model actually sees.

In [43]:
with_det = df[df["n_detections"] > 0]["image"].tolist()
without_det = df[df["n_detections"] == 0]["image"].tolist()

images_to_annotate = with_det[:3] + without_det[:2]
name_to_path = {p.name: p for p in image_paths}

for img_name in images_to_annotate:
    img_path = name_to_path[img_name]
    model.predict(
        source=str(img_path),
        conf=0.25,
        device=device,
        save=True,
        project=str(OUTPUT_DIR),
        name="annotated",
        exist_ok=True,
        verbose=False
    )

print(f"Annotated images saved to {ANNOTATED_DIR}")
print(f"Examples: {images_to_annotate}")

Results saved to D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\results\gas_bottle_baseline\annotated
Results saved to D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\results\gas_bottle_baseline\annotated
Results saved to D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\results\gas_bottle_baseline\annotated
Results saved to D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\results\gas_bottle_baseline\annotated
Results saved to D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\results\gas_bottle_baseline\annotated
Annotated images saved to d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\results\gas_bottle_baseline\annotated
Examples: ['14_55_back_left_cropped_0001.jpg', '14_55_back_left_cropped_0002.jpg', '14_55_back_left_cropped_0003.jpg', '14_55_back_left_cropped_0000.jpg', '14_55_back_left_cropped_0004.jpg']


### Summary

In [46]:
summary = {
    "Model": "YOLO11n (pretrained on COCO)",
    "Model size": "nano",
    "Input images": total_images,
    "Image source": str(IMAGE_DIR.relative_to(ROOT)),
    "Confidence threshold": 0.25,
    "Images with >=1 detection": images_with_detections,
    "Percentage with detection": f"{images_with_detections/total_images*100:.1f}%",
    "Total detections": total_detections,
    "Explicit cylinder class in model": has_cylinder_class,
    "Top detected classes": dict(class_counts.most_common(5)),
    "Output CSV": str(CSV_PATH.relative_to(ROOT)),
    "Annotated examples": str(ANNOTATED_DIR.relative_to(ROOT)),
    "Device used": str(device)
}

print("=" * 60)
print("REPORT SUMMARY — BASELINE DETECTION")
print("=" * 60)
for k, v in summary.items():
    print(f"{k:35s}: {v}")
print("=" * 60)

pd.DataFrame([summary]).to_csv(SUMMARY_PATH, index=False)
print(f"\nSummary saved to {SUMMARY_PATH}")

REPORT SUMMARY — BASELINE DETECTION
Model                              : YOLO11n (pretrained on COCO)
Model size                         : nano
Input images                       : 321
Image source                       : data\all_images
Confidence threshold               : 0.25
Images with >=1 detection          : 248
Percentage with detection          : 77.3%
Total detections                   : 454
Explicit cylinder class in model   : False
Top detected classes               : {'vase': 128, 'person': 97, 'chair': 32, 'bottle': 31, 'fire hydrant': 18}
Output CSV                         : results\gas_bottle_baseline\baseline_detections.csv
Annotated examples                 : results\gas_bottle_baseline\annotated
Device used                        : cpu

Summary saved to d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\results\gas_bottle_baseline\baseline_summary.csv
